In [1]:
pip install numpy matplotlib pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ---- palette (site colours) ----
BG    = "#06070E"
WHITE = "#DCE8E9"
CYAN  = "#E9EAEC"

rng = np.random.default_rng(7)

# ---- 4:5 portrait canvas ----
W, H, DPI = 7.2, 9.0, 100           # -> 720 x 900 px
fig, ax = plt.subplots(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

xlim, ylim = (-4.2, 4.2), (-5.2, 5.2)
gx = np.linspace(*xlim, 180)
gy = np.linspace(*ylim, 220)
GX, GY = np.meshgrid(gx, gy)

# ---- edge feather: fade the outer frame into the background ----
bg_rgb = np.array([6, 7, 14]) / 255          # #06070E
FEATHER = 0.18                               # fraction of each side that fades
_oy, _ox = 320, 256
_ny = np.linspace(0, 1, _oy)[:, None]
_nx = np.linspace(0, 1, _ox)[None, :]
def _smooth(t):
    t = np.clip(t, 0, 1); return t*t*(3 - 2*t)
_vis = (_smooth(np.minimum(_nx, 1-_nx)/FEATHER) *
        _smooth(np.minimum(_ny, 1-_ny)/FEATHER))
_overlay = np.zeros((_oy, _ox, 4))
_overlay[..., :3] = bg_rgb
_overlay[..., 3] = 1 - _vis                  # opaque bg at edges, clear in centre

N_FRAMES, N_PART, FPS = 240, 120, 12
base_modes = np.array([[-1.9, 2.6], [2.2, 0.5], [-0.6, -3.0]])

def gauss(mx, my, sx, sy, rho):
    a, b = (GX - mx)/sx, (GY - my)/sy
    q = (a*a - 2*rho*a*b + b*b)/(1 - rho*rho)
    return np.exp(-0.5*q)/(2*np.pi*sx*sy*np.sqrt(1 - rho*rho))

def params(theta):
    e = 0.5*(1 + np.cos(theta))                   # 1 at loop seam (collapsed), 0 mid (spread)
    drift = np.array([0.9*np.sin(theta), 1.1*np.sin(2*theta)])
    logits = np.array([2.8*e, -1.1*e, -1.7*e])
    w = np.exp(logits); w /= w.sum()
    sx = 2.5*(1-e) + 0.40*e
    sy = 3.0*(1-e) + 0.52*e
    rho = 0.35*np.sin(theta)
    modes = base_modes*(1 - 0.45*e) + drift
    m0 = modes[0].copy()
    modes = modes*(1 - e) + m0*e                  # all modes merge onto one as e -> 1
    return e, w, modes, sx, sy, rho

def grad_log(pts, modes, w, sx, sy):
    comp, dens = [], np.zeros(len(pts))
    for (mx, my), wi in zip(modes, w):
        ax_, ay_ = (pts[:,0]-mx)/sx, (pts[:,1]-my)/sy
        d = wi*np.exp(-0.5*(ax_*ax_ + ay_*ay_))/(2*np.pi*sx*sy)
        comp.append((d, mx, my)); dens += d
    dens = np.maximum(dens, 1e-12)
    g = np.zeros_like(pts)
    for d, mx, my in comp:
        r = d/dens
        g[:,0] += r*(-(pts[:,0]-mx)/sx**2)
        g[:,1] += r*(-(pts[:,1]-my)/sy**2)
    return g

# ---- smooth, periodic noise -> seamless loop ----
def periodic_noise(K, n):
    spec = np.fft.rfft(rng.standard_normal((K, n, 2)), axis=0)
    freqs = np.arange(spec.shape[0])
    spec *= np.exp(-(freqs/(K*0.10))**2)[:, None, None]    # low-pass -> smooth motion
    x = np.fft.irfft(spec, n=K, axis=0)
    x /= (x.std(axis=0, keepdims=True) + 1e-9)
    return x

noise = periodic_noise(N_FRAMES, N_PART)

# ---- simulate: burn in onto the periodic orbit, record one clean loop ----
step, LOOPS = 0.05, 3
P = rng.uniform([xlim[0], ylim[0]], [xlim[1], ylim[1]], size=(N_PART, 2))
positions = np.zeros((N_FRAMES, N_PART, 2))
for t in range(LOOPS*N_FRAMES):
    f = t % N_FRAMES
    e, w, modes, sx, sy, rho = params(2*np.pi*f/N_FRAMES)
    temp = 0.85*(1-e) + 0.05*e
    P = P + step*grad_log(P, modes, w, sx, sy) + np.sqrt(2*step*temp)*noise[f]
    P[:,0] = np.clip(P[:,0], *xlim); P[:,1] = np.clip(P[:,1], *ylim)
    if t >= (LOOPS-1)*N_FRAMES:
        positions[f] = P

def draw(frame):
    e, w, modes, sx, sy, rho = params(2*np.pi*frame/N_FRAMES)
    Z = np.zeros_like(GX)
    for (mx, my), wi in zip(modes, w):
        Z += wi*gauss(mx, my, sx, sy, rho)
    zmax = Z.max()
    P, Pp = positions[frame], positions[frame-1]   # frame-1 at 0 -> last frame, seamless

    ax.clear(); ax.set_facecolor(BG)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis("off")
    ax.contour(GX, GY, Z, levels=zmax*np.array([0.05,0.12,0.22]),
               colors=CYAN, linewidths=0.8, alpha=0.30)
    ax.contour(GX, GY, Z, levels=zmax*np.array([0.38,0.58,0.80]),
               colors=CYAN, linewidths=1.2, alpha=0.80)
    ax.contour(GX, GY, Z, levels=zmax*np.array([0.92]),
               colors=WHITE, linewidths=1.3, alpha=0.35 + 0.5*e)
    ax.plot([Pp[:,0], P[:,0]], [Pp[:,1], P[:,1]], color=CYAN, lw=0.6, alpha=0.14)
    ax.scatter(P[:,0], P[:,1], s=5, color=CYAN, alpha=0.5, linewidths=0)
    ax.scatter([modes[0,0]], [modes[0,1]], s=30+140*e,
               color=WHITE, alpha=0.06+0.24*e, linewidths=0)
    ax.imshow(_overlay, extent=(xlim[0], xlim[1], ylim[0], ylim[1]),
              origin="lower", aspect="auto", zorder=20, interpolation="bilinear")
    return []

anim = FuncAnimation(fig, draw, frames=N_FRAMES, interval=1000/FPS, blit=False)
anim.save("hero_2.gif", writer=PillowWriter(fps=FPS))
plt.close(fig)
print("Saved hero_2.gif  (%d frames, %.1fs loop)" % (N_FRAMES, N_FRAMES/FPS))

Saved hero_2.gif  (240 frames, 20.0s loop)
